# Experiment: AKI Data Cleaning

Objective:
- Load `mimic_aki_cohort_raw.csv` exported from BigQuery.
- Review missingness, separate identifiers / timing / target columns, and remove obviously unusable columns.
- Preserve `subject_id` for later subject-level model splitting.


In [3]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

INPUT_CSV = Path('outputs/mimic_aki_cohort_raw.csv')
OUTPUT_DIR = Path('outputs')
CLEANED_CSV = OUTPUT_DIR / 'mimic_aki_cohort_cleaned.csv'
DATA_DICTIONARY_CSV = OUTPUT_DIR / 'mimic_aki_data_dictionary.csv'
TARGET_COL = 'future_aki_24h'
ID_COLS = ['subject_id', 'hadm_id', 'stay_id']
TIME_COLS = [
    'icu_intime',
    'anchor_time',
    'obs_window_start',
    'obs_window_end',
    'pred_window_start',
    'pred_window_end',
]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print({'input_csv': str(INPUT_CSV), 'cleaned_csv': str(CLEANED_CSV)})


{'input_csv': 'outputs/mimic_aki_cohort_raw.csv', 'cleaned_csv': 'outputs/mimic_aki_cohort_cleaned.csv'}


## Load raw dataset

This notebook is intended for Google Colab after `data_fetch_aki.py` has exported the raw AKI cohort CSV.


In [4]:
df = pd.read_csv(INPUT_CSV)
print(f'Raw shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
print('Target prevalence:')
print(df[TARGET_COL].value_counts(dropna=False).sort_index())
df.head()


Raw shape: 29,274 rows x 165 columns
Target prevalence:
future_aki_24h
0    12156
1    17118
Name: count, dtype: int64


,subject_id,hadm_id,stay_id,icu_intime,anchor_time,obs_window_start,obs_window_end,pred_window_start,pred_window_end,future_aki_24h,...,spo2_last_6h,spo2_min_6h,spo2_max_6h,spo2_mean_6h,spo2_count_6h,urine_total_6h,urine_count_6h,weight_used_for_uo_norm,urine_rate_6h,oliguria_like_6h
0,12466550,23998182,30000153,2174-09-29 12:09:00,2174-09-29 18:09:00,2174-09-29 12:09:00,2174-09-29 18:09:00,2174-09-29 18:09:00,2174-09-30 18:09:00,1,...,99.0,99.0,100.0,99.833333,6.0,470.0,5.0,70.0,1.119048,0.0
1,12207593,22795209,30000646,2194-04-29 01:39:22,2194-04-29 07:39:22,2194-04-29 01:39:22,2194-04-29 07:39:22,2194-04-29 07:39:22,2194-04-30 07:39:22,0,...,92.0,78.0,100.0,96.076923,26.0,700.0,1.0,NaN,NaN,NaN
2,12168737,29283664,30001336,2186-03-20 00:44:48,2186-03-20 06:44:48,2186-03-20 00:44:48,2186-03-20 06:44:48,2186-03-20 06:44:48,2186-03-21 06:44:48,0,...,97.0,95.0,99.0,96.750000,8.0,1600.0,4.0,77.0,3.463203,0.0
3,10682002,20035892,30003087,2132-12-01 20:58:25,2132-12-02 02:58:25,2132-12-01 20:58:25,2132-12-02 02:58:25,2132-12-02 02:58:25,2132-12-03 02:58:25,0,...,90.0,90.0,100.0,97.428571,7.0,NaN,NaN,69.0,NaN,0.0
4,16165135,24791729,30003125,2116-04-02 20:07:00,2116-04-03 02:07:00,2116-04-02 20:07:00,2116-04-03 02:07:00,2116-04-03 02:07:00,2116-04-04 02:07:00,0,...,97.0,95.0,98.0,96.750000,8.0,350.0,2.0,73.2,0.796903,0.0


## Basic inspection

Start with a compact summary of column types, unique counts, and missingness.


In [5]:
column_summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'n_unique': df.nunique(dropna=False),
    'missing_count': df.isna().sum(),
})
column_summary['missing_pct'] = (column_summary['missing_count'] / len(df) * 100).round(2)
column_summary.sort_values(['missing_pct', 'n_unique'], ascending=[False, True]).head(20)


,dtype,n_unique,missing_count,missing_pct
wbc_count_6h,float64,2,29233,99.86
wbc_delta_6h,float64,2,29233,99.86
wbc_first_6h,float64,36,29233,99.86
wbc_last_6h,float64,36,29233,99.86
wbc_min_6h,float64,36,29233,99.86
wbc_max_6h,float64,36,29233,99.86
wbc_mean_6h,float64,36,29233,99.86
lactate_count_6h,float64,11,15129,51.68
lactate_min_6h,float64,141,15129,51.68
lactate_last_6h,float64,148,15129,51.68


## Missingness review

High-missingness columns are not automatically leakage, but they are good candidates for pruning before baseline models.


In [6]:
missingness = (
    df.isna()
      .mean()
      .mul(100)
      .sort_values(ascending=False)
      .rename('missing_pct')
      .reset_index()
      .rename(columns={'index': 'column'})
)
missingness.head(30)


,column,missing_pct
0,wbc_min_6h,99.859944
1,wbc_first_6h,99.859944
2,wbc_last_6h,99.859944
3,wbc_delta_6h,99.859944
4,wbc_count_6h,99.859944
5,wbc_mean_6h,99.859944
6,wbc_max_6h,99.859944
7,lactate_first_6h,51.680672
8,lactate_last_6h,51.680672
9,lactate_min_6h,51.680672


## Column grouping

Separate identifiers, timing columns, the target, and candidate feature columns. Keep `subject_id` so model splitting can remain subject-level.


In [7]:
missing_threshold = 95.0
required_keep_cols = set(ID_COLS + TIME_COLS + [TARGET_COL])

high_missing_cols = [
    col for col, pct in missingness.set_index('column')['missing_pct'].items()
    if pct >= missing_threshold and col not in required_keep_cols
]
constant_cols = [
    col for col in df.columns
    if df[col].nunique(dropna=False) <= 1 and col not in required_keep_cols
]
explicit_leakage_cols = [
    col for col in [
        'icu_outtime',
        'dischtime',
        'los_icu',
        'los_hospital',
        'hospital_expire_flag',
        'aki_onset_time',
        'aki_onset_stage',
        'aki_stage_max',
        'has_aki_anytime',
        'eligible_for_prediction',
    ]
    if col in df.columns
]

feature_candidate_cols = [
    col for col in df.columns
    if col not in required_keep_cols and col not in explicit_leakage_cols
]

print('ID columns:', ID_COLS)
print('Time columns:', TIME_COLS)
print('Target column:', TARGET_COL)
print(f'Feature candidate count: {len(feature_candidate_cols)}')
print('High missing columns:', high_missing_cols)
print('Constant columns:', constant_cols)
print('Explicit leakage review columns:', explicit_leakage_cols)


ID columns: ['subject_id', 'hadm_id', 'stay_id']
Time columns: ['icu_intime', 'anchor_time', 'obs_window_start', 'obs_window_end', 'pred_window_start', 'pred_window_end']
Target column: future_aki_24h
Feature candidate count: 155
High missing columns: ['wbc_min_6h', 'wbc_first_6h', 'wbc_last_6h', 'wbc_delta_6h', 'wbc_count_6h', 'wbc_mean_6h', 'wbc_max_6h']
Constant columns: ['first_icu_stay', 'first_hosp_stay']
Explicit leakage review columns: []


## Cleaning decisions

This first-pass cleaned file removes obviously unusable columns while preserving identifiers, timing metadata, and the target. Modeling notebooks can further exclude timing columns from features without losing auditability.


In [8]:
drop_cols = sorted(set(high_missing_cols + constant_cols + explicit_leakage_cols) - {'subject_id'})
cleaned_df = df.drop(columns=drop_cols, errors='ignore').copy()

for col in TIME_COLS:
    if col in cleaned_df.columns:
        cleaned_df[col] = pd.to_datetime(cleaned_df[col], errors='coerce')

cleaned_df = cleaned_df.sort_values(['subject_id', 'stay_id']).reset_index(drop=True)
print(f'Cleaned shape: {cleaned_df.shape[0]:,} rows x {cleaned_df.shape[1]:,} columns')
print('Dropped columns:', drop_cols)
cleaned_df.head()


Cleaned shape: 29,274 rows x 156 columns
Dropped columns: ['first_hosp_stay', 'first_icu_stay', 'wbc_count_6h', 'wbc_delta_6h', 'wbc_first_6h', 'wbc_last_6h', 'wbc_max_6h', 'wbc_mean_6h', 'wbc_min_6h']


,subject_id,hadm_id,stay_id,icu_intime,anchor_time,obs_window_start,obs_window_end,pred_window_start,pred_window_end,future_aki_24h,...,spo2_last_6h,spo2_min_6h,spo2_max_6h,spo2_mean_6h,spo2_count_6h,urine_total_6h,urine_count_6h,weight_used_for_uo_norm,urine_rate_6h,oliguria_like_6h
0,10001725,25563031,31205490,2110-04-11 15:52:22,2110-04-11 21:52:22,2110-04-11 15:52:22,2110-04-11 21:52:22,2110-04-11 21:52:22,2110-04-12 21:52:22,0,...,98.0,96.0,100.0,98.833333,6.0,300.0,1.0,72.2,0.692521,0.0
1,10001884,26184834,37510196,2131-01-11 04:20:05,2131-01-11 10:20:05,2131-01-11 04:20:05,2131-01-11 10:20:05,2131-01-11 10:20:05,2131-01-12 10:20:05,0,...,100.0,90.0,100.0,98.000000,7.0,625.0,4.0,65.0,1.602564,0.0
2,10002013,23581541,39060235,2160-05-18 10:00:53,2160-05-18 16:00:53,2160-05-18 10:00:53,2160-05-18 16:00:53,2160-05-18 16:00:53,2160-05-19 16:00:53,1,...,100.0,100.0,100.0,100.000000,2.0,280.0,3.0,96.0,0.486111,1.0
3,10002155,23822395,33685454,2129-08-04 12:45:00,2129-08-04 18:45:00,2129-08-04 12:45:00,2129-08-04 18:45:00,2129-08-04 18:45:00,2129-08-05 18:45:00,1,...,92.0,92.0,97.0,93.428571,7.0,250.0,1.0,53.0,0.786164,0.0
4,10002348,22725460,32610785,2112-11-30 23:24:00,2112-12-01 05:24:00,2112-11-30 23:24:00,2112-12-01 05:24:00,2112-12-01 05:24:00,2112-12-02 05:24:00,0,...,95.0,93.0,97.0,94.571429,7.0,50.0,1.0,41.6,0.200321,1.0


## Export cleaned dataset

The cleaned CSV remains row-level at `stay_id` grain and keeps `subject_id` for subject-level splitting in the modeling notebook.


In [9]:
cleaned_df.to_csv(CLEANED_CSV, index=False)

data_dictionary = pd.DataFrame({
    'column': cleaned_df.columns,
    'dtype': cleaned_df.dtypes.astype(str).values,
    'missing_pct': cleaned_df.isna().mean().mul(100).round(2).values,
})
data_dictionary.to_csv(DATA_DICTIONARY_CSV, index=False)

print(f'Saved cleaned dataset to {CLEANED_CSV}')
print(f'Saved data dictionary to {DATA_DICTIONARY_CSV}')


Saved cleaned dataset to outputs/mimic_aki_cohort_cleaned.csv
Saved data dictionary to outputs/mimic_aki_data_dictionary.csv


## Next steps

- Review the dropped-column list before training.
- Keep subject-level splits by `subject_id` in downstream modeling.
- If the first Colab run reveals unexpected sparsity, adjust thresholds here rather than redefining the target.
